# Inspect and evaluate a saved WiFi model

Select **one** historical run. No CNN training, data discovery, or random re-splitting is performed.
The saved manifest determines recordings, labels, model architecture and exact segment order.

- For saved metrics and training curves only: run through **Training history**, then stop. This does not load signal data or use the GPU.
- For fresh evaluation: continue through restoration, export, scoring, calibration and prediction. A new `conformal-*` directory preserves older results.
- Embedding projections are optional. Background diagnostics use the saved heads without training any model.
- Restart the kernel before selecting a different run, especially after GPU training in another session.

Memory: NPY signals are memory-mapped on CPU; only one small inference batch enters the device, with gradients disabled. Checkpoints stay on CPU. Embeddings and scores still occupy **system RAM**; projection sampling limits computation, but compressed NPZ exports must be decompressed in RAM. Reduce `inference.batch_size` in baseline.yaml (or evaluation YAML) to 2 or 1 if CUDA runs out of memory, or choose `device_name = "cpu"`. A small batch cannot free GPU memory held by another notebook/kernel.

In [ ]:
from pathlib import Path
import json

project_root = None  # Set the project directory if launching elsewhere.
PROJECT_ROOT = Path(project_root or Path.cwd()).expanduser().resolve()
if (PROJECT_ROOT / "step02_models").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
PYTHON_ROOT = PROJECT_ROOT / "python"
import sys
if str(PYTHON_ROOT) not in sys.path:
    sys.path.insert(0, str(PYTHON_ROOT))
NPY_ROOT = PYTHON_ROOT / "dataset"
available_runs = sorted(p.parent for p in (PYTHON_ROOT / "runs").glob("*/best.pt")
                        if (p.parent / "experiment.json").is_file())
for p in available_runs:
    print(p.name)

In [ ]:
run_path = "20261005T045346921595Z"  # REQUIRED: e.g. "20261001T192314897806Z", or an absolute run directory.
checkpoint_name = "best.pt"  # Another compatible checkpoint in the same run is also supported.
saved_evaluation = None  # None selects the latest completed conformal-*; or give its directory name.
evaluation_config_path = None  # Optional YAML override for inference/score/calibration/weighting/alpha_grid.
device_name = "auto"  # "auto", "cuda", or "cpu"
run_embedding_projections = True
log_to_wandb = False

## Open the selected run and inspect saved results

In [3]:
# Invalidate downstream results before rerunning a prerequisite.
evaluation = None
if run_path is None:
    raise ValueError("Set run_path to one of the run directories listed above, then rerun this cell.")
run_dir = Path(run_path).expanduser()
if not run_dir.is_absolute():
    run_dir = PYTHON_ROOT / (run_dir if run_dir.parts[0] == "runs" else Path("runs") / run_dir)
run_dir = run_dir.resolve()
checkpoint_path = run_dir / checkpoint_name
if not checkpoint_path.is_file():
    raise FileNotFoundError(checkpoint_path)
experiment = json.loads((run_dir / "experiment.json").read_text())
result = (json.loads((run_dir / "history.json").read_text())
          if (run_dir / "history.json").exists() else {"history": [], "best_epoch": None})
saved_dirs = sorted(p.parent for p in run_dir.glob("conformal-*/metrics.json")
                    if (p.parent / "config.json").exists())
if saved_evaluation is None:
    saved_dir = saved_dirs[-1] if saved_dirs else None
else:
    saved_dir = Path(saved_evaluation).expanduser()
    if not saved_dir.is_absolute():
        saved_dir = run_dir / saved_dir
    if saved_dir.resolve().parent != run_dir:
        raise ValueError("Choose a saved evaluation belonging to the selected run")
print("Run:", run_dir)
print("Checkpoint:", checkpoint_path.name)
print("Classes:", experiment["class_names"])
print("Saved split sizes:", {k: len(v) for k, v in experiment["splits"].items()})
print("Completed evaluations:", [p.name for p in saved_dirs])
if saved_dir is not None:
    print("Saved results (not a new evaluation):", saved_dir)
    saved_provenance = json.loads((saved_dir / "config.json").read_text())
    print("Saved result checkpoint:", saved_provenance.get("checkpoint"),
          "epoch:", saved_provenance.get("checkpoint_epoch"))
    print((saved_dir / "metrics.json").read_text())
else:
    print("No completed evaluation saved yet. Continue below to evaluate this checkpoint.")

Run: /home/kailong/Desktop/WiFi_Conformal_Prediction/python/runs/20261001T192314897806Z
Checkpoint: best.pt
Classes: ['doc', 'sit', 'squat', 'walk', 'wipe']
Saved split sizes: {'train': 2356, 'validation': 335, 'calibration': 672, 'test': 373, 'reference': 537}
Completed evaluations: []
No completed evaluation saved yet. Continue below to evaluate this checkpoint.


## Training history
Read-only: the existing training plot is not overwritten.

In [ ]:
import matplotlib.pyplot as plt

history = result["history"]
epochs = [r["epoch"] for r in history]
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, metric in zip(axes, ("loss", "accuracy")):
    for split in ("train", "validation"):
        ax.plot(epochs, [r[f"{split}_{metric}"] for r in history], label=split)
    if result.get("best_epoch") is not None:
        ax.axvline(result["best_epoch"], color="gray", linestyle="--", label="best epoch")
    ax.set(xlabel="Epoch", ylabel=metric.capitalize())
    ax.legend()
axes[2].plot(epochs, [r["learning_rate"] for r in history])
axes[2].set(xlabel="Epoch", ylabel="Learning rate")
fig.tight_layout()
# Inspection only: preserve the saved training plot.
plt.show()
plt.close(fig)

## Restore evaluation settings
Use the selected saved CP configuration, or the run's config.yaml when no saved evaluation is selected. An optional YAML overrides explicitly supplied evaluation fields. Current-format manifests/configs are required; no legacy migration or historical fallback is performed. The current baseline controls the inference memory budget unless explicitly overridden.

Only saved network weights are restored. Score fitting uses training representations; no NN training or new data splitting occurs.

In [ ]:
# Invalidate downstream results before rerunning a prerequisite.
evaluation = None
import sys
import copy
import numpy as np
import torch
from dataclasses import asdict
if str(PYTHON_ROOT) not in sys.path:
    sys.path.insert(0, str(PYTHON_ROOT))
from step00_config.schema import InferenceConfig
from step00_config.schema import ModelConfig, AdversarialConfig, EncoderBackgroundConfig, EncoderTaskConfig, DecoderConfig
from step02_models.build import build_model

if device_name not in ("auto", "cpu", "cuda"):
    raise ValueError("device_name must be auto, cpu or cuda")
device = torch.device(("cuda" if torch.cuda.is_available() else "cpu") if device_name == "auto" else device_name)
# Read only evaluation sections; training defaults must not reinterpret saved splits.
from step07_evaluation.settings import restore_evaluation_settings
config, settings_source = restore_evaluation_settings(run_dir, saved_dir, evaluation_config_path)
inference_batch_size = InferenceConfig(**config["inference"]).batch_size
split_config = experiment["split"]
split_seed, training_seed = experiment["split_seed"], experiment["training_seed"]
class_names = experiment["class_names"]
time_shape, stft_shape = experiment["time_shape"], experiment["stft_shape"]
model_config = ModelConfig(**experiment["model"])
encoder_task_config = EncoderTaskConfig(**experiment["encoder_task"])
adversarial_config = AdversarialConfig(**encoder_task_config.adversarial)
domain_names = experiment["domain_names"]
encoder_background_config = EncoderBackgroundConfig(**experiment["encoder_background"])
decoder_config = DecoderConfig(**experiment["decoder"])
data_config = experiment["data"]
monitor_config = copy.deepcopy(experiment.get("monitoring", {}))
monitor_config["enabled"] = log_to_wandb
print("Evaluation settings source:", settings_source)
print("Device:", device, "Inference batch size:", inference_batch_size)
print(json.dumps({k: config[k] for k in ("score", "calibration", "alpha_grid", "weighting", "inference")}, indent=2))

## Rebuild loaders from the saved manifest
Restore the exact train/validation/calibration/test membership and row order. No cache conversion or new splitting is performed.

In [ ]:
recordings = experiment["recordings"]
files = [NPY_ROOT / r["time"] for r in recordings]
stft_files = [NPY_ROOT / r["stft"] for r in recordings]
targets = np.asarray([r["label"] for r in recordings], dtype=np.int64)
metadata = [r["metadata"] for r in recordings]
# Restore exactly the saved four-way split.
if set(experiment["splits"]) != {"train", "validation", "calibration", "test"}:
    raise ValueError("Expected current four-way split manifest")
split_samples = {k: np.asarray(experiment["splits"][k], dtype=np.int64).reshape(-1, 2)
                 for k in ("train", "validation", "calibration", "test")}
if split_config["mode"] == "domain_holdout":
    domains = np.asarray([m[data_config["domain_key"]] for m in metadata])
for name, pairs in split_samples.items():
    if not len(pairs):
        raise ValueError(f"Empty saved split: {name}")
    if (pairs < 0).any() or (pairs[:, 0] >= len(recordings)).any():
        raise ValueError(f"Invalid saved indices: {name}")
    for i in np.unique(pairs[:, 0]):
        for path, shape in ((files[i], time_shape), (stft_files[i], stft_shape)):
            a = np.load(path, mmap_mode="r", allow_pickle=False)
            if list(a.shape) != [recordings[i]["segments"], *shape] or a.dtype != np.float32:
                raise ValueError(f"Cache shape/dtype differs from saved run: {path}")
            del a
        if pairs[pairs[:, 0] == i, 1].max() >= recordings[i]["segments"]:
            raise ValueError(f"Segment outside saved recording: {name}, {i}")

from step01_preprocessing.dataset import NpySegments

datasets = {name: NpySegments(files, stft_files, targets, pairs) for name, pairs in split_samples.items()}
print({name: len(ds) for name, ds in datasets.items()})

## Restore the selected model
CPU checkpoint loading avoids a second GPU copy of the weights. No optimizer is created.
Background activity adversarial conditioning is controlled independently by `encoder_background.adversarial.conditional`. When enabled, the head receives the true training-domain one-hot; task prediction and CP do not require this condition. Conditional activity diagnostics skip unknown domains and report the evaluated sample count.

The activity adversary width is controlled independently by `encoder_background.adversarial.hidden_dim`.

Both adversaries have independent optimizer/scheduler settings. `data.domain_key` is shared by both branches, CL, sampling and weighting. `decoder` controls reconstruction and shares the main optimizer. Each encoder difference loss detaches the opposite embedding; both are inactive without background.


In [ ]:
# Invalidate downstream results before rerunning a prerequisite.
evaluation = None
# Restart the kernel before changing runs to discard old GPU references.
selected_checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
checkpoint_model = selected_checkpoint["model_config"]
if asdict(ModelConfig(**checkpoint_model)) != asdict(model_config):
    raise ValueError("Checkpoint architecture differs from experiment manifest")
checkpoint_background = EncoderBackgroundConfig(**selected_checkpoint["encoder_background"])
checkpoint_decoder = DecoderConfig(**selected_checkpoint["decoder"])
if asdict(checkpoint_background) != asdict(encoder_background_config):
    raise ValueError("Checkpoint background configuration differs from experiment manifest")
if asdict(checkpoint_decoder) != asdict(decoder_config):
    raise ValueError("Checkpoint decoder differs from experiment manifest")
model = build_model(model_config, time_shape, stft_shape, len(class_names), training_seed,
                      domain_classes=len(domain_names) if adversarial_config.enabled else 0,
                      domain_hidden_dim=adversarial_config.hidden_dim,
                      domain_conditional=adversarial_config.conditional,
                      encoder_background=encoder_background_config, background_domains=len(domain_names), decoder=decoder_config)
model.load_state_dict(selected_checkpoint["model_state_dict"], strict=True)
model = model.to(device).eval()
print("Checkpoint epoch:", selected_checkpoint["epoch"], "Validation loss:", selected_checkpoint.get("validation_loss"))
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
del selected_checkpoint

## Export embeddings

Export each saved split once, in original row order, using inference.batch_size. All inputs are clean; training augmentation is not applied. The cached task/background representations are shared by score fitting, target weight estimation and prediction. Target labels are used only for evaluation metrics.

In [ ]:
from evaluation_pipeline import ConformalEvaluation

## Fit the score model

Choose a score model and its settings here. The restored `config["score"]` selects the scorer. KDE uses a per-class density on the model embedding. SVM/HBGB are alternatives using probability-margin scores. All scores have shape [samples, classes]; larger values mean less conformity. Point predictions use the minimum score, not the maximum calibrated p-value.

`score.kind: softmax` directly uses `1 - p_y` from the frozen NN probabilities and skips intermediate model fitting. For pooled ordinary split CP, set `weighted_cp: false`; calibration pools all true-label scores for any score choice.

Fit KDE/SVM/HBGB only on training embeddings. Use validation accuracy to inspect/choose settings, then freeze them before running Step 06. No additional embedding normalization is applied. KDE bandwidth is in embedding units. The predeclared alpha grid below is for reporting, not selecting a threshold from test performance.
SVM probabilities use sigmoid calibration with five seeded stratified folds inside the training split (`CalibratedClassifierCV`, `ensemble=False`). Each training class needs at least five samples. This probability calibration is separate from Step 06 rank calibration.

In [ ]:
# Recreate the session after changing evaluation settings.
evaluation = None
evaluation = ConformalEvaluation(model, datasets, experiment, config, checkpoint_path,
                                  settings_source=settings_source, aps_seed=config.get("aps_seed", 42))

In [ ]:
evaluation.export()

In [ ]:
evaluation.fit_score()

## Calibrate the score model

Run this after model/score settings have been selected with validation. Calibration uses only the score of each sample's true class. Pooled calibration uses all true-label scores in one reference distribution. Ties are included, with the finite-sample +1 correction.

Each invocation creates a separate conformal result directory. Do not tune the model, scoring settings or alpha using calibration/test results. These are segment-level empirical evaluations: segments from one recording can be dependent, so file-disjoint splitting alone does not establish exchangeability of individual segments or a distribution-free coverage guarantee.

Randomized APS is calibrated alongside the configured score: unweighted APS always runs, and weighted APS uses the same calibration/test weights when weighted_cp is enabled. Calibration and test use independent random streams; draws stay fixed across the alpha sweep. No forced nonempty sets are added.

In [ ]:
evaluation.calibrate()

## Predict sets on test data

For candidate class c, p_c = (1 + number of reference scores >= candidate score) / (reference count + 1). Include c exactly when p_c > alpha. These p-values are not class probabilities and need not sum to one. Test labels are used only for subsequent evaluation, not scoring or set construction.
With `weighted_cp: true`, each count is replaced by a sum of weights, and the +1 is replaced by the candidate test point's own weight. Score ties remain included. Weights estimated from reused unlabeled target inputs do not establish an exact target coverage guarantee.

In [ ]:
evaluation.predict()

## Evaluate and save results

Coverage includes empty prediction sets. Report the scorer's point-classification accuracy separately from CNN accuracy. Per-class results and the fixed alpha sweep are descriptive evaluations; do not use test outcomes to select a configuration. Every test segment receives equal weight, so recordings with more segments contribute more to these metrics.

The common coverage/set-size plot compares configured CP, APS unweighted, and APS weighted when enabled, with nominal coverage and NN accuracy. All results and APS draws are saved together in the new conformal directory.

In [ ]:
cp_dir = None
cp_dir = evaluation.save_results()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

evaluation.require_saved_results()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, rows in evaluation.method_results.items():
    levels = [row["alpha"] for row in rows]
    axes[0].plot(levels, [row["coverage"] for row in rows], "o-", label=name)
    axes[1].plot(levels, [row["mean_set_size"] for row in rows], "o-", label=name)
axes[0].plot(levels, 1 - np.asarray(levels), "--", color="gray", label="Nominal 1-alpha")
axes[0].axhline(evaluation.nn_accuracy, color="tab:red", linestyle="-.", label=f"NN accuracy = {evaluation.nn_accuracy:.3f}")
axes[0].set(xlabel="Alpha", ylabel="Coverage / accuracy", ylim=(0, 1.02))
axes[1].set(xlabel="Alpha", ylabel="Mean prediction-set size", ylim=(0, len(evaluation.class_names)))
for ax in axes:
    ax.legend(fontsize=8)
fig.suptitle("CP and randomized APS comparison")
fig.tight_layout()
fig.savefig(evaluation.cp_dir / "coverage_set_size.png", dpi=150)
plt.show()
plt.close(fig)

In [ ]:
from step07_evaluation.monitoring import log_evaluation
evaluation.require_saved_results()
log_evaluation(evaluation, monitor_config)

## Compare embedding projections

1. Load and sample saved embeddings, keeping labels and recording/segment identifiers aligned.
2. Choose methods and parameters, compute coordinates, and save NPZ files.
3. Adjust plot styling and save PNG files using the cached coordinates. Rerun only this cell to change colors, markers, legends or layout.

The default methods are t-SNE and metric MDS; Isomap and Spectral Embedding are optional entries in the computation cell. Reduce `viz_max_per_split` when pairwise-distance methods are expensive. Graph disconnection warnings matter; projections alone do not establish domain invariance.

In [ ]:
from step07_evaluation.projections import load_projection_data

viz_dir = cp_dir  # Or a saved conformal-* directory.
viz_seed = 42
viz_max_per_split = 1000  # None keeps every segment.
# Invalidate old coordinates before changing the selected data.
viz_data = projections = None
if run_embedding_projections:
    viz_data = load_projection_data(viz_dir, viz_seed, viz_max_per_split)

In [ ]:
from sklearn.manifold import TSNE, Isomap, SpectralEmbedding, MDS
import json
import numpy as np

# Changing these parameters requires rerunning this cell; style changes do not.
projections = None
if viz_data is not None:
    embeddings = viz_data["embeddings"]
    map_neighbors = min(30, len(embeddings) - 1)
    embedding_maps = {
        "tsne": TSNE(n_components=2, perplexity=min(30.0, len(embeddings) - 1),
                     init="pca", learning_rate="auto", random_state=viz_data["seed"]),
        # "isomap": Isomap(n_components=2, n_neighbors=map_neighbors),
        # "spectral": SpectralEmbedding(n_components=2, n_neighbors=map_neighbors,
        #                               affinity="nearest_neighbors", random_state=viz_data["seed"]),
        "mds": MDS(n_components=2, n_init=4, max_iter=300,
                   init="random", eps=1e-6, random_state=viz_data["seed"]),
    }
    computed = {}
    for name, mapper in embedding_maps.items():
        print(f"Computing {name} for {len(embeddings)} samples...", flush=True)
        coordinates = mapper.fit_transform(embeddings)
        if name == "mds" and mapper.n_iter_ >= mapper.max_iter:
            print("MDS reached max_iter; increase max_iter to check convergence.")
        computed[name] = coordinates
        np.savez_compressed(viz_data["directory"] / f"embedding_{name}.npz",
                            coordinates=coordinates, labels=viz_data["labels"],
                            domains=viz_data["domains"], splits=viz_data["splits"],
                            recording=viz_data["recording"], segment_index=viz_data["segment_index"],
                            parameters=json.dumps(mapper.get_params()), seed=viz_data["seed"],
                            max_per_split=-1 if viz_data["max_per_split"] is None else viz_data["max_per_split"])
    projections = computed
else:
    print("Projection skipped.")


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# Edit layout, colors, markers and legends here. Cached coordinates are reused.
map_titles = {"tsne": "t-SNE", "isomap": "Isomap", "spectral": "Spectral Embedding", "mds": "Metric MDS"}
if projections is not None:
    for name, coordinates in projections.items():
        fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharex=True, sharey=True)
        for ax, values, title in zip(axes, (viz_data["labels"], viz_data["domains"], viz_data["splits"]),
                                    (f"Task: {viz_data['task']}", f"Domain: {viz_data['domain_key']}", "Data split")):
            categories = list(dict.fromkeys(values))
            colors = plt.get_cmap("tab20", len(categories))
            for i, category in enumerate(categories):
                selected = values == category
                ax.scatter(*coordinates[selected].T, s=12, alpha=0.65,
                           color=colors(i), label=category, linewidths=0)
            ax.set(title=title, xlabel=f"{map_titles[name]} 1")
            ax.xaxis.set_major_locator(MaxNLocator(4))
            ax.legend(fontsize=8, markerscale=1.5)
        axes[0].set_ylabel(f"{map_titles[name]} 2")
        fig.suptitle(f"{map_titles[name]} | n={len(coordinates)}")
        fig.tight_layout()
        output_path = viz_data["directory"] / f"embedding_{name}.png"
        fig.savefig(output_path, dpi=180)
        plt.show()
        plt.close(fig)
        print("Saved:", output_path)
else:
    print("No completed projections. Run the loading and computation cells first if needed.")
